# Format historical data for analyses

## Imports

In [10]:
import pandas as pd
import numpy as np
import datetime as dt

## Shiller's historical S&P 500 data

This data is more or less raw.

In [11]:
df = pd.read_excel('../../data/shiller/shiller_sp500.xls', sheet_name='Data', header=None, skiprows=8)

### Manually adjust columns names

There's some weird formatting due to a preamble above the first several columns and some unused columns.  I chose slightly shorter names here for convenience and drop the columns with no actual data.

In [12]:
dropped_columns = ["No Data 1", "No Data 2"]

columns = [
    "Date",
    "S&P",
    "Dividend",
    "Earnings",
    "CPI",
    "Date Fraction",
    "Long Interest Rate",
    "Real Price",
    "Real Dividend",
    "Real Total Return Price",
    "Real Earnings",
    "Real TR Scaled Earnings",
    "CAPE",
    dropped_columns[0],
    "TR CAPE",
    dropped_columns[1],
    "Excess CAPE Yield",
    "Monthly Bond Returns",
    "Monthly Real Bond Returns",
    "10 Year Real Stock Return",
    "10 Year Real Bond Return",
    "10 Year Excess Return"
]

df.columns = columns

# Sanity check
filled_dropped_rows = df[~df[dropped_columns[0]].isna() | ~df[dropped_columns[1]].isna()]
print("Was there any data in the dropped columns?", len(filled_dropped_rows) > 0)

# Drop the empty columns
df = df.drop(columns=dropped_columns)

Was there any data in the dropped columns? False


### Drop trailing row that contains text instead of numeric entries

The last row for some entries describes the extrapolation methodology.  These will confuse plots/analyses, so they're dropped.

In [13]:
last_index = len(df) - 1
df = df.drop(last_index)

### Format the 'Date' column

Shiller's data formats dates as `<year>.<month>`, where January, 2005 corresponds to 2005.01 and October, 1994 coresponds to 1994.1.  Plotting and other comparisons are easier to perform using python's `datetime.datetime` type.

In [14]:
def to_datetime(date: np.float64) -> dt.datetime:
    """
    Convert values in the 'Date' column of the Shiller data into datetime.datetime format.
    """
    return dt.datetime(to_year(date), to_month(date), 1)  # Assume 1st of month

def to_year(date: np.float64) -> int:
    """
    Extract the year from 'Date' column values in the Shiller data.
    """
    return int(date)

def to_month(date: np.float64) -> int:
    """
    Extract the month, [1,12], from 'Date' column values in the Shiller data.
    """
    month = (date - int(date)) / .0099  # Handle floating point rounding
    return int(month)

In [15]:
df['Date'] = df['Date'].apply(to_datetime)

### Export this formatted data to a .csv

In [16]:
csv_path = '../../data/shiller/shiller_sp500_pandas_formatted.csv'
df.to_csv(csv_path, index=False)
test_df = pd.read_csv(csv_path, parse_dates=['Date'])

In [17]:
# Can't do a simple size comparison for the datetime subtraction, so remove dates
df_nodate = df.drop(columns=['Date'])
test_df_nodate = test_df.drop(columns=['Date'])

diff_df = (df_nodate - test_df_nodate)
diff_df = diff_df.dropna()
same_data = diff_df[diff_df > 1E-10].dropna().size < 1
print(f"Imported dataframe has the same data columns: {same_data}")


# Dates should be identical
same_dates = df['Date'].equals(test_df['Date'])
print(f"Imported dataframe has the same date column: {same_dates}")


print("--------------------------------------------------")


print(f"Exported csv creates equivalent dataframe: {same_data and same_dates}")

Imported dataframe has the same data columns: True
Imported dataframe has the same date column: True
--------------------------------------------------
Exported csv creates equivalent dataframe: True
